In [0]:
# Databricks notebook source
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "churn_dev")
catalog, schema = dbutils.widgets.get("catalog"), dbutils.widgets.get("schema")

import mlflow, mlflow.sklearn
mlflow.set_registry_uri("databricks-uc")
from pyspark.sql import functions as F

model_uri = f"models:/{catalog}.{schema}.churn_model@champion"
model = mlflow.sklearn.load_model(model_uri)

pdf = spark.table(f"{catalog}.{schema}.gold_churn_features").toPandas()
feats = ["contract", "internet_service", "tenure_months", "monthly_charges",
         "support_calls", "total_spend", "calls_per_year", "is_month_to_month"]
pdf["churn_probability"] = model.predict_proba(pdf[feats])[:, 1]

out = (spark.createDataFrame(pdf[["customer_id", "churn_probability"]])
       .withColumn("model_uri", F.lit(model_uri))
       .withColumn("scored_at", F.current_timestamp()))
out.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.churn_scores")